In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
print(v_file_date)

In [0]:
#Creamos una tabla sql con spark
spark.sql("""
          create table if not exists movie_gold.results_movie
          (
            year_release_date INT,
            country_name string,
            company_name string,
            budget FLOAT,
            revenue FLOAT,
            movie_id INT,
            country_id INT,
            company_id INT,
            created_date date,
            updated_date date
            )
            using delta
          """)

In [0]:
#la vista sera nuestra tabla source

spark.sql(f"""
            create or replace temp view v_results_movie 
            as
            select m.year_release_date, c.country_name, pco.company_name, m.budget, m.revenue,
                   m.movie_id, c.country_id, pco.company_id
            from movie_silver.movies m 
                 inner join movie_silver.productions_countries pc on m.movie_id = pc.movie_id
                 inner join movie_silver.countries c on pc.country_id = c.country_id
                 inner join movie_silver.movies_companies mc on m.movie_id = mc.movie_id
                 inner join movie_silver.productions_companies pco on mc.company_id = pco.company_id
            where m.file_date = '{v_file_date}'
          """)

In [0]:
spark.sql("""
            MERGE INTO movie_gold.results_movie AS A
            USING v_results_movie AS B
            ON (A.movie_id = B.movie_id and A.country_id = B.country_id and A.company_id = B.company_id )
            WHEN MATCHED THEN
                UPDATE SET 
                    A.year_release_date = B.year_release_date,
                    A.country_name = B.country_name,
                    A.company_name = B.company_name,
                    A.budget = B.budget,
                    A.revenue = B.revenue,
                    A.updated_date = current_timestamp
            WHEN NOT MATCHED
                THEN INSERT(A.year_release_date, A.country_name, A.company_name, A.budget, A.revenue, A.movie_id, A.country_id, A.company_id, A.created_date)
                VALUES(B.year_release_date, B.country_name, B.company_name, B.budget, B.revenue, B.movie_id, B.country_id, B.company_id, current_timestamp)
         """)